In [ ]:
# ==============================================================================
# КЛЕТКА 1. Конфигурация путей, модели и параметров вычислительного ядра (v3.2)
# ==============================================================================
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = next(
    (p for p in (_cwd, _cwd.parent, _cwd.parent.parent) if (p / "data").exists() or (p / "src" / "v2_engine.py").is_file()),
    _cwd,
)
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# В папке data лежат папки городов (Алматы, куляб, модель семей, Орша, тенино, хорезмская) и файл онтологии .ttl
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR
REPORTS_DIR = PROJECT_ROOT / "reports"

# Автопоиск папки моделей: приоритет training/models (где лежат gguf/, llama_cpp/, LoRA/SFT)
_candidate_model_dirs = [
    PROJECT_ROOT / "training" / "models",
    PROJECT_ROOT / "training_epanet_ai" / "models",
    PROJECT_ROOT.parent / "training_epanet_ai" / "models",
    _cwd / "training_epanet_ai" / "models",
    PROJECT_ROOT / "models",
]
EXTRA_MODEL_DIRS = []
for _md in _candidate_model_dirs:
    if _md.exists() and _md.is_dir() and _md not in EXTRA_MODEL_DIRS:
        EXTRA_MODEL_DIRS.append(_md)
MODELS_DIR = EXTRA_MODEL_DIRS[0] if EXTRA_MODEL_DIRS else (PROJECT_ROOT / "training" / "models")

# Автопоиск файла онтологии в корне data/ (приоритет: aquabit_hydraulic_ontology_v3.ttl, иначе первый *.ttl)
ONTOLOGY_TTL_PATH = DATA_DIR / "aquabit_hydraulic_ontology_v3.ttl"
if not ONTOLOGY_TTL_PATH.exists() and DATA_DIR.exists():
    _ttl_files = sorted(DATA_DIR.glob("*.ttl"))
    if _ttl_files:
        ONTOLOGY_TTL_PATH = _ttl_files[0]

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Целевой город или путь к папке с выгрузкой ПК «Аквабит» (приоритет: Report.json + *.digWater, fallback: CSV + *.inp)
# Доступные папки городов в data/: Алматы, куляб, модель семей, Орша, тенино, хорезмская
TARGET_CITY_OR_PATH = "модель семей"
REPORT_JSON_PATH = None   # Опционально: явный путь к Report.json (иначе автопоиск в папке города)
DIGWATER_PATH = None      # Опционально: явный путь к *.digWater (иначе автопоиск в папке города)
GEOMETRY_INP_PATH = None  # Опционально: legacy fallback путь к .inp, если .digWater отсутствует
SP31_PDF_PATH = None      # Опционально: путь к PDF/TXT СП 31.13330 (иначе автопоиск в data/ + встроенная база)
QDRANT_PATH = DATA_DIR / "qdrant_sp31"  # Локальное постоянное хранилище Qdrant для чанков СП 31.13330
ENABLE_LLM_SELF_CORRECTION = True  # Автоматический 1-шаговый прогон самокоррекции LLM при критических нарушениях

# Детализация временной шкалы на карте (48 шагов = интервал 30 мин в формате ЧЧ:ММ без меток R360)
TIMELINE_BUCKETS = 48

# Выбор локальной GGUF-модели для ИИ-диагностики:
#   "auto_local"          — (по умолчанию) автоматически берет уже скачанный .gguf из training_epanet_ai/models/gguf/
#   "Qwen2.5-7B-Instruct" — принудительно Qwen2.5-7B-Instruct-Q4_K_M.gguf (при отсутствии скачает с HF)
#   "Qwen3.5-4B"          — принудительно Qwen3.5-4B-Q4_K_M.gguf (при отсутствии скачает с HF)
MODEL_PRESET = "auto_local"
LOCAL_GGUF_OVERRIDE = None  # Можно указать имя конкретного файла из gguf/, например: "Qwen3.5-4B-Q4_K_M.gguf"

if MODEL_PRESET == "Qwen3.5-4B":
    GGUF_REPO = "unsloth/Qwen3.5-4B-GGUF"
    GGUF_FILENAME = "Qwen3.5-4B-Q4_K_M.gguf"
else:
    GGUF_REPO = "bartowski/Qwen2.5-7B-Instruct-GGUF"
    GGUF_FILENAME = "Qwen2.5-7B-Instruct-Q4_K_M.gguf"

LLAMA_SERVER_PATH = None  # Заполнять не нужно, если llama-server.exe лежит в training_epanet_ai/models/llama_cpp/
LLAMA_BACKEND = "cuda"    # "cuda" или "cpu"
LLAMA_N_GPU_LAYERS = 99
LLAMA_CONTEXT = 12288     # 12288 отлично помещается в VRAM с Qwen3.5-4B (2.55 ГБ) и дает запас под полный отчет
INFERENCE_MAX_NEW_TOKENS = 2800

# Сколько самых критичных строк в каждой таблице (узлы, трубы, насосы) сразу заполнять живыми выводами ИИ при сборке отчета
# (0 = отключить пакетную генерацию при сборке; остальные строки в HTML всегда можно заполнить по кнопке «✨ ИИ»)
TABLE_AI_MAX_ROWS_PER_TAB = 10

# Оставлять ли llama-server запущенным на порту 8080 после генерации отчета
# (позволяет кнопкам «Сгенерировать развернутый анализ графика (ИИ)» и «✨ ИИ» в строках таблиц обращаться к живой модели)
KEEP_LLAMA_SERVER_FOR_HTML = True
LLAMA_HTML_PORT = 8080

def find_llama_server_binary() -> Path | None:
    import shutil
    if LLAMA_SERVER_PATH:
        p = Path(LLAMA_SERVER_PATH)
        if p.exists() and p.is_file():
            return p
    search_roots = [
        MODELS_DIR / "llama_cpp",
        *EXTRA_MODEL_DIRS,
        PROJECT_ROOT / "training_epanet_ai",
        PROJECT_ROOT.parent / "training_epanet_ai",
        PROJECT_ROOT / "tools",
        PROJECT_ROOT / "bin",
        PROJECT_ROOT,
    ]
    seen_roots = set()
    candidates = []
    seen_files = set()
    for root in search_roots:
        if not root or not root.exists() or str(root) in seen_roots:
            continue
        seen_roots.add(str(root))
        for name in ("llama-server.exe", "llama-server"):
            for hit in root.rglob(name):
                if hit.is_file() and ".venv" not in hit.parts:
                    rp = str(hit.resolve())
                    if rp not in seen_files:
                        seen_files.add(rp)
                        candidates.append(hit)
    if candidates:
        want_cuda = str(globals().get("LLAMA_BACKEND", "cuda")).lower() == "cuda"
        def _bin_rank(p: Path):
            s = str(p).lower()
            has_cuda_dll = any(p.parent.glob("*cuda*.dll")) or any(p.parent.glob("*cudart*.dll")) or any(p.parent.glob("*cublas*.dll"))
            is_cuda = has_cuda_dll or any(k in s for k in ("cuda", "cu12", "cu11", "gpu"))
            is_cpu = ("cpu" in s) and not is_cuda
            if want_cuda:
                return (0 if is_cuda else (2 if is_cpu else 1), len(str(p)))
            return (0 if is_cpu else 1, len(str(p)))
        return sorted(candidates, key=_bin_rank)[0]
    w = shutil.which("llama-server") or shutil.which("llama-server.exe")
    return Path(w) if w else None

def find_all_local_gguf_models() -> list[Path]:
    hits = []
    seen = set()
    roots = [*EXTRA_MODEL_DIRS, MODELS_DIR, PROJECT_ROOT / "training_epanet_ai", PROJECT_ROOT.parent / "training_epanet_ai"]
    for root in roots:
        if not root or not root.exists():
            continue
        for f in root.rglob("*.gguf"):
            if not f.is_file():
                continue
            rp = str(f.resolve())
            # Пропускаем мультимодальные проекторы (mmproj) — это не текстовые модели
            if "mmproj" in f.name.lower() or rp in seen:
                continue
            seen.add(rp)
            hits.append(f)
    return sorted(hits, key=lambda p: p.stat().st_size, reverse=True)

def resolve_selected_gguf_model() -> Path | None:
    local_models = find_all_local_gguf_models()
    if LOCAL_GGUF_OVERRIDE:
        ov = Path(LOCAL_GGUF_OVERRIDE)
        if ov.exists() and ov.is_file():
            return ov
        for f in local_models:
            if f.name.lower() == str(LOCAL_GGUF_OVERRIDE).lower() or str(LOCAL_GGUF_OVERRIDE).lower() in f.name.lower():
                return f
    if MODEL_PRESET != "auto_local":
        for f in local_models:
            if f.name.lower() == GGUF_FILENAME.lower():
                return f
        return None
    # Режим auto_local: берем лучшую из уже лежащих в training_epanet_ai/models/gguf/
    if not local_models:
        return None
    def _rank(p: Path):
        n = p.name.lower()
        if "qwen2.5-7b" in n or "qwen2_5_7b" in n:
            return (0, -p.stat().st_size)
        if "epanet" in n or "aquabit" in n:
            return (1, -p.stat().st_size)
        if "qwen3.5" in n or "qwen3" in n:
            return (2, -p.stat().st_size)
        if "qwen" in n:
            return (3, -p.stat().st_size)
        return (4, -p.stat().st_size)
    return sorted(local_models, key=_rank)[0]

available_cities = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()]) if DATA_DIR.exists() else []
_found_llama = find_llama_server_binary()
_local_ggufs = find_all_local_gguf_models()
_selected_gguf = resolve_selected_gguf_model()
print(f"Конфигурация v3.2 загружена. Режим модели: {MODEL_PRESET}")
print(f"Корень проекта: {PROJECT_ROOT} | Города в data/: {available_cities}")
print(f"Папка моделей (MODELS_DIR): {MODELS_DIR}")
print(f"Онтология: {ONTOLOGY_TTL_PATH} ({'найдена' if ONTOLOGY_TTL_PATH.exists() else 'НЕ НАЙДЕНА'})")
print(f"llama-server: {_found_llama if _found_llama else 'НЕ НАЙДЕН (укажите LLAMA_SERVER_PATH в Клетке 1!)'}")
if _local_ggufs:
    print("Найденные локальные .gguf модели:")
    for _idx, _gf in enumerate(_local_ggufs, 1):
        _mark = "  <-- ВЫБРАНА" if (_selected_gguf and _gf.resolve() == _selected_gguf.resolve()) else ""
        print(f"   {_idx}. {_gf.name} ({_gf.stat().st_size / (1024**3):.2f} ГБ) [{_gf.parent}]{_mark}")
else:
    print(f"Локальные .gguf не найдены в {MODELS_DIR} (при запуске Клетки 4 скачается {GGUF_FILENAME})")


In [ ]:
# ==============================================================================
# КЛЕТКА 2. Аналитическое ядро Аквабит v3.2, онтология и генератор отчетов
# ==============================================================================
import html
import io
import math
import os
import re
import sys
import time
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd

# Подключаем src/ и корень проекта в sys.path
_cwd = Path.cwd()
for _root in [globals().get("PROJECT_ROOT"), _cwd, _cwd.parent, _cwd.parent.parent]:
    if not _root:
        continue
    _rp = Path(_root)
    for _p in (_rp / "src", _rp):
        if _p.exists() and str(_p) not in sys.path:
            sys.path.insert(0, str(_p))

# Модули src подключаются через совместимые интерфейсы.
# После изменения кода в src перезапустите ядро Jupyter и выполните ячейки заново.
import ontology_engine as _oe
import v2_engine as _ve
from v2_report_and_align import render_standalone_html_report, generate_reference_pure_facts_report
from ontology_engine import (
    load_hydraulic_ontology, evaluate_ontology_rules, analyze_problem_domains,
    build_enriched_ai_context, build_ontology_ai_system_prompt, extract_ai_dashboards_and_content,
)
from v2_engine import (
    process_aquabit_city_fast, process_aquabit_city_v3,
    load_report_json, parse_digwater_metadata_and_topology, read_object_names,
    parse_inp_metadata_and_topology, read_aquabit_csv, find_col,
)
from processing.identifiers import clean_object_id
from llm.post_validator import (
    sanitize_ai_report_text, validate_ai_report_against_facts, build_self_correction_prompt,
)
from eval.metrics import evaluate_report_quality_metrics
from rag.retriever import get_default_normative_retriever, find_sp31_document

print(f"Аналитическое ядро Аквабит v3.2 загружено из: {Path(_oe.__file__).parent}")


In [ ]:
# ==============================================================================
# КЛЕТКА 3. Генератор интерактивного HTML5-отчета v3.2
# ==============================================================================
from pathlib import Path
from typing import Any

def render_aquabit_html_v3(
    facts: dict[str, Any],
    charts: dict[str, Any],
    main_md: str,
    parsed_dashboards: dict[str, Any],
    out_html: Path,
    gen_sec: float = 0.0,
) -> str:
    """Собирает автономный интерактивный HTML5-отчет v3.2:
    - Сортировка и фильтрация во всех 4 таблицах + автофильтры из карточек уведомлений и текста ИИ (filter:tab:preset)
    - Сквозная связка График -> Таблица -> Карта (с пульсацией выбранного объекта)
    - Исправленный зум карты от центра экрана без серого экрана
    - Детальная шкала времени в формате ЧЧ:ММ без (R360) с мгновенным обновлением паспорта выбранного объекта
    - Живой контекстный ИИ-анализ графиков и выбранных участков таблиц по кнопке.
    """
    return render_standalone_html_report(
        facts=facts,
        chart_payload=charts,
        ai_markdown=main_md,
        output_html_path=out_html,
        generation_meta={"inference_sec": gen_sec},
        selected_dashboards_spec=parsed_dashboards,
    )

print("Функция сборки интерактивного отчета render_aquabit_html_v3 (v3.2) инициализирована.")


In [ ]:
# ==============================================================================
# КЛЕТКА 4. Запуск сквозного расчета: граф-диагностика, TTL-онтология и генерация отчета
# ==============================================================================
import shutil
import socket
import subprocess
import tempfile
import time
import json
import re
import unicodedata
import requests
from pathlib import Path
from typing import Any
from huggingface_hub import hf_hub_download
from IPython.display import display, HTML

_ACTIVE_LLAMA_PROC = None

def resolve_city_directory(city_or_path: str | Path) -> Path:
    """Находит папку города по структуре data/<город> (с учетом регистра и Unicode NFC/NFD)."""
    p = Path(city_or_path)
    if p.exists() and p.is_dir():
        return p
    target_norm = unicodedata.normalize("NFC", str(city_or_path)).strip().lower()
    search_roots = [
        RAW_DATA_DIR,
        PROJECT_ROOT / "data",
        PROJECT_ROOT / "data" / "raw",
    ]
    for root in search_roots:
        direct = root / str(city_or_path)
        if direct.exists() and direct.is_dir():
            return direct
        if root.exists() and root.is_dir():
            for sub in root.iterdir():
                if sub.is_dir() and unicodedata.normalize("NFC", sub.name).strip().lower() == target_norm:
                    return sub
    avail = [sub.name for sub in (PROJECT_ROOT / "data").iterdir() if sub.is_dir()] if (PROJECT_ROOT / "data").exists() else []
    raise FileNotFoundError(
        f"Папка города '{city_or_path}' не найдена в {RAW_DATA_DIR}. Доступные папки в data/: {avail}"
    )

def run_pipeline_v3(city_or_path: str | Path):
    global _ACTIVE_LLAMA_PROC
    t0 = time.perf_counter()
    ontology = load_hydraulic_ontology(ONTOLOGY_TTL_PATH)
    city_dir = resolve_city_directory(city_or_path)
    inp_file = Path(GEOMETRY_INP_PATH) if globals().get("GEOMETRY_INP_PATH") else None
    json_file = Path(REPORT_JSON_PATH) if globals().get("REPORT_JSON_PATH") else None
    dw_file = Path(DIGWATER_PATH) if globals().get("DIGWATER_PATH") else None

    # Инициализируем RAG-слой СП 31.13330 в Qdrant (с переиспользованием локального хранилища)
    sp31_doc = Path(SP31_PDF_PATH) if globals().get("SP31_PDF_PATH") else find_sp31_document()
    rag_ret = get_default_normative_retriever(
        pdf_path=sp31_doc,
        qdrant_path=globals().get("QDRANT_PATH"),
    )
    print(
        f"📚 RAG СП 31.13330: Qdrant={'активен' if rag_ret.qdrant_active else 'fallback in-memory'} | "
        f"чанков={len(rag_ret.points)} | файл={sp31_doc.name if sp31_doc else 'встроенная база'}"
    )

    print(f"[1/3] Обработка данных ({city_dir}), трассировка графа смежности и оценка правил TTL-онтологии...")
    facts, charts = process_aquabit_city_v3(
        city_dir,
        inp_path=inp_file,
        report_json_path=json_file,
        digwater_path=dw_file,
        timeline_buckets=globals().get("TIMELINE_BUCKETS", 48),
    )
    facts["chart_payload"] = charts
    print(f"   Источник данных модели: {facts.get('data_source_type', 'unknown')} (имена объектов: {'да' if facts.get('has_inp_names') else 'нет'})")

    triggered_rules = evaluate_ontology_rules(facts, ontology)
    problem_domains = analyze_problem_domains(facts, triggered_rules, ontology=ontology)

    print(f"Ключевая проблемная область сети: «{problem_domains['primary_domain']}»")
    for d in problem_domains["ranked_domains"]:
        print(f"   • [{d['score']:.1f} б.] {d['title']} ({d['details']})")

    # Обогащенный причинно-следственный контекст с конкретными ID (Источник -> Насос -> Коллектор -> Труба)
    enriched_context = build_enriched_ai_context(facts, problem_domains["ranked_domains"], triggered_rules)

    print("[2/3] Формирование однозначного аналитического заключения ИИ по графу смежности...")
    system_prompt = build_ontology_ai_system_prompt(ontology)
    user_prompt = (
        f"Выполни глубокий технический аудит гидравлической модели города {facts['city_name']} "
        "на основе предрассчитанных причинно-следственных цепочек графа (Источник -> Насос -> Коллектор -> Отходящая труба). "
        "Обязательно используй кликабельные ссылки `ID` и фильтр-ссылки на таблицы вида (см. [описание](filter:tab:preset)):\n"
        + json.dumps(enriched_context, ensure_ascii=False, separators=(",", ":"))
    )

    raw_ai_text = ""
    gen_sec = 0.0
    ai_source_label = ""

    exe = find_llama_server_binary() if "find_llama_server_binary" in globals() else None
    if not exe:
        print("⚠️ ВНИМАНИЕ: Бинарник llama-server.exe НЕ НАЙДЕН! Нейросеть не может быть запущена.")
        print(f"   Искали в: {MODELS_DIR}, {PROJECT_ROOT / 'tools'}, {PROJECT_ROOT} и системном PATH.")
        print(r"   Решение: задайте точный путь в Клетке 1 -> LLAMA_SERVER_PATH = r'C:\path\to\llama-server.exe'")
    else:
        try:
            local_direct_model = resolve_selected_gguf_model() if "resolve_selected_gguf_model" in globals() else next(iter(MODELS_DIR.rglob(GGUF_FILENAME)), None)
            if local_direct_model:
                model_file = local_direct_model
            else:
                print(f"⬇️ Локальный .gguf не найден, скачивание {GGUF_REPO} / {GGUF_FILENAME} в {MODELS_DIR / 'gguf'}...")
                model_file = Path(
                    hf_hub_download(
                        repo_id=GGUF_REPO,
                        filename=GGUF_FILENAME,
                        local_dir=str(MODELS_DIR / "gguf" / GGUF_REPO.replace("/", "--"))
                    )
                )
            size_gb = round(model_file.stat().st_size / (1024 ** 3), 2)
            keep_alive = bool(globals().get("KEEP_LLAMA_SERVER_FOR_HTML", True))
            port = int(globals().get("LLAMA_HTML_PORT", 8080)) if keep_alive else 0
            if port == 0:
                with socket.socket() as s:
                    s.bind(("127.0.0.1", 0))
                    port = s.getsockname()[1]
            base_url = f"http://127.0.0.1:{port}"
            ngl_val = LLAMA_N_GPU_LAYERS if LLAMA_BACKEND.lower() == "cuda" else 0
            log_path = REPORTS_DIR / "llama_server.log"

            print(f"🚀 Запуск локальной нейросети: {model_file.name} ({size_gb} ГБ)")
            print(f"   Путь к весам: {model_file}")
            print(f"   Бинарник: {exe} | backend={LLAMA_BACKEND}, -ngl={ngl_val}, -c={LLAMA_CONTEXT}, порт={port}")

            already_running = False
            try:
                if requests.get(base_url + "/health", timeout=1).status_code == 200:
                    already_running = True
                    print(f"♻️ Используется уже запущенный llama-server на {base_url} (чтобы сменить модель, выполните stop_background_llama_server() в Клетке 5)")
            except requests.RequestException:
                pass

            def _launch_server(use_jinja: bool):
                cmd = [
                    str(exe), "-m", str(model_file),
                    "--host", "127.0.0.1", "--port", str(port),
                    "-c", str(LLAMA_CONTEXT),
                    "-ngl", str(ngl_val),
                ]
                if use_jinja:
                    cmd.append("--jinja")
                lf = open(log_path, "w", encoding="utf-8", errors="replace")
                p = subprocess.Popen(
                    cmd,
                    cwd=str(exe.parent),
                    stdout=lf,
                    stderr=subprocess.STDOUT,
                    creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0)
                )
                return p, lf

            proc = None
            log_fh = None
            ready = already_running
            if not already_running:
                for attempt_jinja in (True, False):
                    proc, log_fh = _launch_server(use_jinja=attempt_jinja)
                    _ACTIVE_LLAMA_PROC = proc
                    for _ in range(120):
                        if proc.poll() is not None:
                            break
                        try:
                            if requests.get(base_url + "/health", timeout=1).status_code == 200:
                                ready = True
                                break
                        except requests.RequestException:
                            pass
                        time.sleep(1.0)
                    if ready:
                        break
                    if log_fh:
                        log_fh.close()
                        log_fh = None
                    err_tail = log_path.read_text(encoding="utf-8", errors="replace")[-1200:] if log_path.exists() else ""
                    if attempt_jinja and ("jinja" in err_tail.lower() or "unknown argument" in err_tail.lower()):
                        print("ℹ️ Текущая сборка llama-server не поддерживает --jinja, перезапуск без этого флага...")
                        continue
                    print(f"❌ llama-server завершился с ошибкой (код {proc.returncode}). Последние строки {log_path}:")
                    print(err_tail)
                    break

            try:
                if ready:
                    # Узнаем реальный n_ctx запущенного сервера и точное число токенов промпта
                    active_n_ctx = int(LLAMA_CONTEXT)
                    try:
                        pr = requests.get(base_url + "/props", timeout=2)
                        if pr.ok:
                            pj = pr.json()
                            dgs = pj.get("default_generation_settings") or {}
                            if dgs.get("n_ctx"):
                                active_n_ctx = int(dgs["n_ctx"])
                    except Exception:
                        pass
                    prompt_tok_est = int((len(system_prompt) + len(user_prompt)) * 0.55)
                    try:
                        tk = requests.post(base_url + "/tokenize", json={"content": system_prompt + "\n" + user_prompt}, timeout=5)
                        if tk.ok:
                            prompt_tok_est = len((tk.json() or {}).get("tokens") or []) + 64
                    except Exception:
                        pass
                    # Гарантируем, что промпт + ответ всегда помещаются в active_n_ctx без HTTP 400
                    reserve_out = min(int(INFERENCE_MAX_NEW_TOKENS), max(1400, active_n_ctx // 3))
                    max_prompt_budget = max(1500, active_n_ctx - reserve_out - 128)
                    if prompt_tok_est > max_prompt_budget:
                        ratio = max(0.35, max_prompt_budget / max(1, prompt_tok_est))
                        keep_chars = int(len(user_prompt) * ratio * 0.92)
                        user_prompt = user_prompt[:keep_chars] + "\n... [контекст автоматически сжат под окно n_ctx]"
                        prompt_tok_est = max_prompt_budget
                    safe_max_tokens = min(int(INFERENCE_MAX_NEW_TOKENS), max(800, active_n_ctx - prompt_tok_est - 96))
                    t_llm = time.perf_counter()
                    print(f"🧠 Отправка графового контекста в GPU-нейросеть (промпт ~{prompt_tok_est} ток. / окно {active_n_ctx} ток., лимит ответа {safe_max_tokens} ток.)...")
                    req_body = {
                        "model": "local",
                        "messages": [
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_prompt}
                        ],
                        "max_tokens": safe_max_tokens,
                        "temperature": 0.20,
                        "top_p": 0.90,
                        "repeat_penalty": 1.12,
                    }
                    if "qwen3" in model_file.name.lower():
                        req_body["chat_template_kwargs"] = {"enable_thinking": False}
                    http_resp = requests.post(
                        base_url + "/v1/chat/completions",
                        json=req_body,
                        timeout=420
                    )
                    if http_resp.ok:
                        rj = http_resp.json()
                        raw_ai_text = rj["choices"][0]["message"]["content"]
                        gen_sec = round(time.perf_counter() - t_llm, 2)
                        usage = rj.get("usage") or {}
                        timings = rj.get("timings") or {}
                        p_tok = usage.get("prompt_tokens", "?")
                        c_tok = usage.get("completion_tokens", "?")
                        tps = timings.get("predicted_per_second")
                        if tps is None and isinstance(c_tok, int) and gen_sec > 0:
                            tps = round(c_tok / gen_sec, 1)
                        else:
                            tps = round(float(tps), 1) if tps else "?"
                        ai_source_label = f"ЖИВАЯ НЕЙРОСЕТЬ ({model_file.name})"
                        print(f"✅ ИИ-отчет сгенерирован ЖИВОЙ НЕЙРОСЕТЬЮ за {gen_sec} сек | Вход: {p_tok} ток., Выход: {c_tok} ток. | Скорость: {tps} ток/с")

                        # Пост-валидация против фактов модели и физических инвариантов + 1-шаговая самокоррекция
                        val_res = validate_ai_report_against_facts(raw_ai_text, facts)
                        if not val_res["is_valid"]:
                            print(f"🛡️ Пост-валидатор выявил замечания ({len(val_res['issues'])}): {val_res['issues'][:3]}")
                            if globals().get("ENABLE_LLM_SELF_CORRECTION", True) and any(
                                v.get("severity") in ("critical", "high") for v in val_res.get("violations", [])
                            ):
                                try:
                                    fix_body = {
                                        "model": "local",
                                        "messages": [
                                            {"role": "system", "content": system_prompt},
                                            {"role": "user", "content": build_self_correction_prompt(raw_ai_text, val_res)},
                                        ],
                                        "max_tokens": safe_max_tokens,
                                        "temperature": 0.15,
                                        "top_p": 0.90,
                                    }
                                    if "qwen3" in model_file.name.lower():
                                        fix_body["chat_template_kwargs"] = {"enable_thinking": False}
                                    fix_resp = requests.post(base_url + "/v1/chat/completions", json=fix_body, timeout=300)
                                    if fix_resp.ok:
                                        fixed_candidate = fix_resp.json()["choices"][0]["message"]["content"]
                                        val_fixed = validate_ai_report_against_facts(fixed_candidate, facts)
                                        if len(val_fixed["issues"]) <= len(val_res["issues"]):
                                            raw_ai_text = val_fixed["corrected_text"]
                                            val_res = val_fixed
                                            print(f"   ✓ Самокоррекция LLM выполнена (осталось замечаний: {len(val_res['issues'])})")
                                        else:
                                            raw_ai_text = val_res["corrected_text"]
                                    else:
                                        raw_ai_text = val_res["corrected_text"]
                                except Exception:
                                    raw_ai_text = val_res["corrected_text"]
                            else:
                                raw_ai_text = val_res["corrected_text"]
                        else:
                            raw_ai_text = val_res["corrected_text"]

                        # Пакетная генерация живых выводов ИИ для критичных строк таблиц (узлы, трубы, насосы)
                        max_rows_tab = int(globals().get("TABLE_AI_MAX_ROWS_PER_TAB", 10))
                        if max_rows_tab > 0:
                            t_tbl = time.perf_counter()
                            print(f"📋 Генерация живых ИИ-выводов для топ-{max_rows_tab} критичных строк в каждой таблице (GPU)...")
                            ai_table_advices = {"nodes": {}, "pipes": {}, "pumps": {}, "valves": {}}
                            pa_f = facts.get("pressure_analysis", {})
                            ppa_f = facts.get("pipes_analysis", {})
                            pma_f = facts.get("pumps_analysis", {})

                            def _clean_cid(val):
                                return clean_object_id(val)

                            def _ask_table_batch(tab_label, items_list):
                                if not items_list:
                                    return {}
                                batch_sys = (
                                    "Ты главный инженер-гидравлик ПК «Аквабит». Верни СТРОГО валидный JSON-объект вида "
                                    "{\"<id>\": \"<краткий инженерный вывод 1-2 предложения до 220 символов: конкретная физическая причина и действие в сети/ПК Аквабит с #ID смежных объектов>\"}. "
                                    "Без markdown-оформления, без пояснений вне JSON, без шаблонных оговорок про задвижки у скважины."
                                )
                                batch_user = f"Таблица: {tab_label}. Сформируй уникальный инженерный вывод для каждого ID:\n" + json.dumps(items_list, ensure_ascii=False, separators=(",", ":"))
                                body = {
                                    "model": "local",
                                    "messages": [{"role": "system", "content": batch_sys}, {"role": "user", "content": batch_user}],
                                    "max_tokens": min(1600, max(450, len(items_list) * 120)),
                                    "temperature": 0.2,
                                    "top_p": 0.9,
                                }
                                if "qwen3" in model_file.name.lower():
                                    body["chat_template_kwargs"] = {"enable_thinking": False}
                                try:
                                    r_b = requests.post(base_url + "/v1/chat/completions", json=body, timeout=180)
                                    if not r_b.ok:
                                        return {}
                                    txt = (r_b.json()["choices"][0]["message"]["content"] or "").strip()
                                    txt = re.sub(r"<think>.*?</think>", "", txt, flags=re.DOTALL).strip()
                                    m_j = re.search(r"\{[\s\S]*\}", txt)
                                    if not m_j:
                                        return {}
                                    parsed_map = json.loads(re.sub(r",\s*([}\]])", r"\1", m_j.group(0)))
                                    if not isinstance(parsed_map, dict):
                                        return {}
                                    valid_ids = {str(it["id"]) for it in items_list}
                                    out_m = {}
                                    for k, v in parsed_map.items():
                                        ck = _clean_cid(str(k).lstrip("#"))
                                        if ck in valid_ids and isinstance(v, str) and len(v.strip()) > 12:
                                            out_m[ck] = sanitize_ai_report_text(v.strip(), facts)
                                    return out_m
                                except Exception:
                                    return {}

                            # 1) Критичные узлы (ошибки P < -3м + худший дефицит + избыток > 60м)
                            node_cands = []
                            seen_nc = set()
                            for src_list in (pa_f.get("worst_data_error_nodes", [])[:5], pa_f.get("worst_deficit_nodes", [])[:5], pa_f.get("top_excess_nodes", [])[:3]):
                                for r in src_list:
                                    cid = _clean_cid(r.get("clean_id") or r.get("node_id"))
                                    if cid and cid not in seen_nc and len(node_cands) < max_rows_tab:
                                        seen_nc.add(cid)
                                        node_cands.append({"id": cid, "z": r.get("elevation_m"), "floors": r.get("floors"), "p_min": r.get("free_pressure_min_m"), "p_max": r.get("free_pressure_max_m"), "dp": r.get("pressure_surplus_deficit_m"), "pipes": (r.get("connected_pipe_ids") or [])[:3]})
                            ai_table_advices["nodes"] = _ask_table_batch("Узлы абонентов", node_cands)

                            # 2) Критичные трубы (макс. удельные потери и узкие места)
                            pipe_cands = []
                            seen_pc = set()
                            for p in ppa_f.get("top_high_loss_pipes", [])[:max_rows_tab]:
                                cid = _clean_cid(p.get("clean_id") or p.get("pipe_id"))
                                if cid and cid not in seen_pc:
                                    seen_pc.add(cid)
                                    pipe_cands.append({"id": cid, "name": p.get("name"), "from": _clean_cid(p.get("start_node")), "to": _clean_cid(p.get("end_node")), "d_mm": p.get("diameter_mm"), "l_m": p.get("length_m"), "v": p.get("velocity_max_m_s"), "i": p.get("headloss_gradient_m_km"), "dh_m": p.get("total_headloss_m"), "c": p.get("hazen_williams_c")})
                            ai_table_advices["pipes"] = _ask_table_batch("Трубопроводы (узкие места)", pipe_cands)

                            # 3) Проблемные насосы (перегрузка + недогрузка)
                            pump_cands = []
                            seen_pmc = set()
                            for src_list in (pma_f.get("overloaded_pumps", [])[:5], pma_f.get("underloaded_pumps", [])[:max_rows_tab]):
                                for p in src_list:
                                    cid = _clean_cid(p.get("clean_id") or p.get("pump_id"))
                                    if cid and cid not in seen_pmc and len(pump_cands) < max_rows_tab:
                                        seen_pmc.add(cid)
                                        pump_cands.append({"id": cid, "type": p.get("pump_type"), "src": _clean_cid(p.get("start_node")), "end": _clean_cid(p.get("end_node")), "end_p": p.get("collector_pressure_m"), "out_pipe": p.get("connected_out_pipe_id"), "out_i": p.get("connected_out_pipe_loss"), "q_w": p.get("q_working_m3_h"), "q_n": p.get("q_nominal_m3_h"), "h_w": p.get("h_working_m")})
                            ai_table_advices["pumps"] = _ask_table_batch("Насосы и скважины", pump_cands)

                            facts["ai_table_advices"] = ai_table_advices
                            tbl_sec = round(time.perf_counter() - t_tbl, 2)
                            gen_sec = round(gen_sec + tbl_sec, 2)
                            print(f"✅ Живые выводы ИИ записаны в таблицы за {tbl_sec} сек: узлы={len(ai_table_advices['nodes'])}, трубы={len(ai_table_advices['pipes'])}, насосы={len(ai_table_advices['pumps'])}")
                    else:
                        print(f"❌ llama-server вернул HTTP {http_resp.status_code}: {http_resp.text[:500]}")
            finally:
                if proc is not None and not keep_alive:
                    proc.terminate()
                    try: proc.wait(timeout=5)
                    except subprocess.TimeoutExpired: proc.kill()
                    if log_fh: log_fh.close()
                elif keep_alive and ready:
                    print(f"🟢 Локальный ИИ-сервер оставлен активным на {base_url} — кнопки живого ИИ в графиках и строках таблиц работают на GPU!")
        except Exception as e:
            print(f"❌ Ошибка запуска локального llama-server: {e}")

    if not raw_ai_text:
        print("⚠️ ВНИМАНИЕ: Нейросеть не сгенерировала текст — сработал резервный Python-генератор по графу (0% GPU)!")
        ai_source_label = "РЕЗЕРВНЫЙ PYTHON-ГЕНЕРАТОР (LLM не запускалась)"
        t_ref = time.perf_counter()
        raw_ai_text = generate_reference_pure_facts_report(
            facts,
            ranked_domains=problem_domains["ranked_domains"],
            ontology=ontology,
        )
        gen_sec = round(time.perf_counter() - t_ref, 2)

    print("[3/3] Сборка интерактивного HTML5-паспорта сети v3.2...")
    main_md, parsed_dashboards = extract_ai_dashboards_and_content(raw_ai_text)
    selected_dashboards = parsed_dashboards.get("section_order") or [d["id"] for d in parsed_dashboards.get("dashboards", [])]
    print(f"ИИ определил ключевые приоритеты анализа: {selected_dashboards}")

    out_html = REPORTS_DIR / f"EPANET_AI_v3_{facts['city_name']}.html"
    rendered_html = render_aquabit_html_v3(facts, charts, main_md, parsed_dashboards, out_html, gen_sec)
    q_metrics = evaluate_report_quality_metrics(main_md, facts, rendered_html)
    print(
        f"📊 Метрики качества ИИ-отчета: Overall={q_metrics['overall_engineering_score']}/100 | "
        f"ID Precision={q_metrics['factual_id_precision']:.2f} | "
        f"Physics={q_metrics['physical_invariant_compliance']:.2f} | "
        f"Norms={q_metrics['normative_grounding_score']:.2f} | "
        f"UI Links={q_metrics['ui_link_integrity']:.2f}"
    )
    total_sec = round(time.perf_counter() - t0, 2)
    print(f"Отчет успешно сформирован за {total_sec} сек (ИИ: {gen_sec} сек).")
    print(f"Путь к HTML-отчету: {out_html.resolve()}")

    rel_path = f"reports/{out_html.name}"
    display(HTML(f'''
    <div style="background:#f8fafc;border:1px solid #cbd5e1;border-radius:10px;padding:16px 20px;margin-top:14px;box-shadow:0 2px 8px rgba(0,0,0,0.04);font-family:sans-serif;">
        <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:10px;">
            <span style="font-weight:700;font-size:15px;color:#0f172a;">Интерактивный отчет ПК «Аквабит» v3.2 ({facts['city_name']})</span>
            <a href="{rel_path}" target="_blank" style="display:inline-block;background:#0284c7;color:#fff;text-decoration:none;padding:7px 16px;border-radius:6px;font-weight:600;font-size:13px;">Открыть отчет в новой вкладке ↗</a>
        </div>
        <p style="margin:0 0 10px;font-size:13px;color:#64748b;">Встроены сортировка и фильтрация таблиц, сквозной переход График → Таблица → Карта, пульсация выбранного объекта, детальная шкала ЧЧ:ММ и живой ИИ-анализ по кнопке.</p>
        <iframe src="{rel_path}" style="width:100%;height:740px;border:1px solid #e2e8f0;border-radius:8px;background:#fff;" frameborder="0"></iframe>
    </div>
    '''))

run_pipeline_v3(TARGET_CITY_OR_PATH)


In [ ]:
# ==============================================================================
# КЛЕТКА 5. Управление фоновым llama-server для живого ИИ-диалога в HTML-отчете
# ==============================================================================
def stop_background_llama_server():
    global _ACTIVE_LLAMA_PROC
    if '_ACTIVE_LLAMA_PROC' in globals() and _ACTIVE_LLAMA_PROC is not None:
        _ACTIVE_LLAMA_PROC.terminate()
        _ACTIVE_LLAMA_PROC = None
        print("🛑 Фоновый llama-server остановлен.")
    else:
        print("ℹ️ Фоновый llama-server сейчас не запущен (HTML-отчет работает в автономном режиме предрассчитанного ИИ-контекста).")

# Раскомментируйте строку ниже, если нужно освободить видеопамять GPU после работы с HTML-отчетом:
# stop_background_llama_server()
